In [ ]:
from pathlib import Path 
import warnings 

import pandas as pd
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from scipy.stats import chi2_contingency

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from xgboost import XGBClassifier

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [ ]:
warnings.filterwarnings('ignore')
RANDOM_STATE = 42 
MAX_ROWS = 200_000
CHUNK_SIZE = 100_000
sns.set_theme(style='whitegrid', palette='deep')

In [ ]:
PROJECT_DIR = Path.cwd()
CSV_PATH = PROJECT_DIR / 'US_Accidents_March23.csv'

if not CSV_PATH.exists():
    raise FileNotFoundError('CSV file not found. Place US_Accidents_March23.csv next to this notebook.')

print(f'CSV found: {CSV_PATH}')
print(f'File size: {CSV_PATH.stat().st_size / (1024 * 1024):.1f} MB')

CSV found: c:\Users\Maina\OneDrive\Desktop\road-accident-severity-kenya\US_Accidents_March23.csv
File size: 2916.5 MB


## 1. Business Understanding

This project asks: can we tell how severe a road accident will be, using only information we'd know at the moment it's reported — not facts we'd only learn afterward? We use a U.S. dataset because it has a usable severity rating, timestamps, weather data, visibility, and road-context details that a Kenyan dataset currently doesn't offer.

The goal isn't to claim U.S. patterns apply directly to Kenya. It's to show how a proper accident-analysis process could be built for a Kenyan road-safety context — while being upfront about what Kenyan data and testing would still be needed.

### Business problem
Road-safety agencies need to know where severe crashes are more likely, so they can prioritize response and investigation. This project aims to show which conditions are linked to more severe accidents, and give a realistic sense of how predictable severity actually is from early information.

### Data-science problem
This is a binary classification task: predict whether an accident is "high severity" or not, based only on information available at the time it's reported — never information that would only be known after the fact.

### Objective
The notebook aims to:
- Describe how severity is distributed in the U.S. data
- Test whether time, weather, visibility, and road context relate to higher severity
- Check how well severity can actually be predicted from early information
- Clearly state what's transferable to Kenya, and what still needs local data

### Success criteria
The project succeeds if it finds real patterns in the U.S. data, evaluates predictions fairly using time-aware testing, and keeps a clear line between "what we found" and "what we can claim applies to Kenya."

### Six analytical questions
1. Which conditions are linked to more severe accidents?
2. Are there clear time-based patterns?
3. What environmental and road factors show up alongside severe accidents?
4. Can severity be predicted from information available at the time of the incident?
5. What information would a Kenyan road-safety system need?
6. Which U.S. variables would need Kenyan equivalents, and what extra Kenyan data would be needed before this could actually be used?

In [ ]:
raw_sample = pd.read_csv(CSV_PATH, nrows=5)

print('Number of columns:', len(raw_sample.columns))
print('Columns:')
for col in raw_sample.columns:
    print(' -', col)

print('\nSample row:')
print(raw_sample.iloc[0].to_dict())